# Neural Network Dreams Bad Apple — Colab anchor ablation

This notebook runs one V4.2 anchor-budget experiment at a time on a hosted Colab GPU while you edit the notebook in VS Code. It does **not** start training when opened.

The Colab kernel cannot read `D:\\Code_archive\\Bad_apple` directly. To keep the repository private, this notebook reads a small source bundle plus the video and frozen autoencoder from Google Drive. From the local repository root, create the source bundle in PowerShell:

```powershell
tar -a -cf bad_apple_code.zip prototype.py requirements.txt README.md neural_bad_apple tests
```

Then put these three files in `MyDrive/neural_bad_apple (1)/assets/`:

- `bad_apple_code.zip`
- `bad_apple.mp4`
- `model_best.pt` (the existing `prototype_runs/basic_full/model_best.pt`)

Checkpoints, metrics, and the per-run `report.md` are written directly to Drive so a Colab disconnect does not erase completed epochs. Training itself uses `/content` for the frame dataset because reading thousands of PNGs directly from Drive is slow.

In [ ]:
# Configuration — edit these values before running the setup cells.
from pathlib import Path

PROJECT_DIR = Path("/content/bad_apple")
DRIVE_ROOT = Path("/content/drive/MyDrive/neural_bad_apple (1)")
CODE_ARCHIVE = DRIVE_ROOT / "assets/bad_apple_code.zip"
VIDEO_SOURCE = DRIVE_ROOT / "assets/bad_apple.mp4"
AUTOENCODER_SOURCE = DRIVE_ROOT / "assets/model_best.pt"

ANCHORS = 220          # Recommended first pass: 0, then 32; 220 already exists locally.
EPOCHS = 12
TRAIN_BATCH_SIZE = 16 # Keep 2 for direct comparability with the local 220-anchor run.
EVAL_BATCH_SIZE = 16
SEED = 7

assert ANCHORS in {0, 16, 32, 55, 110, 220  }, "Choose a staged, not-yet-trained budget."


In [39]:
# Runtime check. In VS Code: Select Kernel -> Colab -> choose the A100 runtime.
import platform

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select a Colab GPU runtime first.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu_name} ({gpu_memory_gib:.1f} GiB)")
if "A100" not in gpu_name.upper():
    print("Warning: this is not an A100; the experiment will still work but take longer.")


Python: 3.13.15
PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)


In [ ]:
# Mount persistent storage. The VS Code extension may open a browser authorization prompt.
import os

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Persistent experiment root: {DRIVE_ROOT}")

# print(os.listdir("/content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/"))
# print(os.listdir(DRIVE_ROOT/ "prototype_runs/anchor_budget_ablation/"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent experiment root: /content/drive/MyDrive/neural_bad_apple (1)
['anchors_000', 'anchors_000_polarity', 'anchors_016', 'anchors_032', 'anchors_032_polarity', 'anchors_055', 'anchors_055_polarity', 'anchors_110', 'anchors_110_polarity', 'anchors_220', 'anchors_220_polarity']


In [65]:
!ls -l "/content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/"

total 44
drwx------ 2 root root 4096 Aug 27 10:20 anchors_000
drwx------ 2 root root 4096 Aug 27 11:22 anchors_000_polarity
drwx------ 2 root root 4096 Aug 27 11:44 anchors_016
drwx------ 2 root root 4096 Aug 27 15:08 anchors_032
drwx------ 2 root root 4096 Aug 27 15:18 anchors_032_polarity
drwx------ 2 root root 4096 Aug 27 16:28 anchors_055
drwx------ 2 root root 4096 Aug 27 16:39 anchors_055_polarity
drwx------ 2 root root 4096 Aug 27 17:51 anchors_110
drwx------ 2 root root 4096 Aug 27 18:02 anchors_110_polarity
drwx------ 2 root root 4096 Aug 27 19:25 anchors_220
drwx------ 2 root root 4096 Aug 27 19:37 anchors_220_polarity


In [58]:
# Extract the private source bundle, normalizing Windows ZIP separators for Linux.
import os
import shutil
import sys
import zipfile

if not CODE_ARCHIVE.is_file():
    raise FileNotFoundError(f"Code bundle not found: {CODE_ARCHIVE}")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CODE_ARCHIVE) as archive:
    for member in archive.infolist():
        normalized = member.filename.replace("\\", "/")
        parts = [part for part in normalized.split("/") if part not in ("", ".")]

        if not parts:
            continue
        if normalized.startswith("/") or ".." in parts:
            raise RuntimeError(f"Unsafe path in code archive: {member.filename}")

        destination = PROJECT_DIR.joinpath(*parts)

        if normalized.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

package_file = PROJECT_DIR / "neural_bad_apple" / "__init__.py"
if not package_file.is_file():
    raise RuntimeError(f"Package was not extracted correctly: {package_file}")

project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
os.environ["PYTHONPATH"] = project_path
os.chdir(PROJECT_DIR)

import neural_bad_apple

print("Project extracted and importable:", neural_bad_apple.__file__)

Project extracted and importable: /content/bad_apple/neural_bad_apple/__init__.py


In [42]:
# Define the command runner and install the project's small dependencies.
import os
import subprocess
from collections import deque

def run(command, *, cwd=None):
    command = [str(part) for part in command]
    print("+", " ".join(command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    if cwd is not None:
        existing_python_path = environment.get("PYTHONPATH", "")
        environment["PYTHONPATH"] = str(cwd) + (os.pathsep + existing_python_path if existing_python_path else "")
    process = subprocess.Popen(
        command, cwd=cwd, env=environment, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    recent_output = deque(maxlen=40)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        recent_output.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        tail = "\n".join(recent_output)
        raise RuntimeError(f"Command exited with code {return_code}. Last output:\n{tail}")

run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=PROJECT_DIR)
run(["python", "-c", "import neural_bad_apple; print('Package import ready:', neural_bad_apple.__file__)"], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Private source ready from {CODE_ARCHIVE} ({CODE_ARCHIVE.stat().st_size / 2**20:.1f} MiB)")


+ python -m pip install -q -r requirements.txt
+ python -c import neural_bad_apple; print('Package import ready:', neural_bad_apple.__file__)
Package import ready: /content/bad_apple/neural_bad_apple/__init__.py
Private source ready from /content/drive/MyDrive/neural_bad_apple (1)/assets/bad_apple_code.zip (0.2 MiB)


In [43]:
# Validate the video and stage the frozen autoencoder on Colab's local disk.
import shutil

if not VIDEO_SOURCE.is_file():
    raise FileNotFoundError(f"Upload the source video to {VIDEO_SOURCE}")
if not AUTOENCODER_SOURCE.is_file():
    raise FileNotFoundError(f"Upload the frozen autoencoder checkpoint to {AUTOENCODER_SOURCE}")

print(f"Video: {VIDEO_SOURCE} ({VIDEO_SOURCE.stat().st_size / 2**20:.1f} MiB)")
run([
    "ffprobe", "-v", "error", "-select_streams", "v:0",
    "-show_entries", "stream=codec_name,width,height,r_frame_rate:format=duration",
    "-of", "default=noprint_wrappers=1", VIDEO_SOURCE,
])

autoencoder_local = PROJECT_DIR / "prototype_runs/basic_full/model_best.pt"
autoencoder_local.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUTOENCODER_SOURCE, autoencoder_local)
print(f"Autoencoder ready: {autoencoder_local}")


Video: /content/drive/MyDrive/neural_bad_apple (1)/assets/bad_apple.mp4 (18.6 MiB)
+ ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate:format=duration -of default=noprint_wrappers=1 /content/drive/MyDrive/neural_bad_apple (1)/assets/bad_apple.mp4
codec_name=h264
width=512
height=384
r_frame_rate=30/1
duration=219.127000
Autoencoder ready: /content/bad_apple/prototype_runs/basic_full/model_best.pt


In [44]:
# Extract the full 219.1-second source locally. This is skipped when all frames already exist.
frame_dir = PROJECT_DIR / "prototype_data/full_source_frames"
frame_count = len(list(frame_dir.glob("frame_*.png"))) if frame_dir.exists() else 0

if frame_count != 6573:
    try:
        run([
            "python", "prototype.py", "extract",
            "--input", VIDEO_SOURCE,
            "--output-dir", frame_dir,
            "--manifest", PROJECT_DIR / "prototype_data/full_manifest.json",
            "--start", "0", "--end", "219.1", "--fps", "30", "--force",
        ], cwd=PROJECT_DIR)
    except RuntimeError as extractor_error:
        print("Project extractor failed; retrying with Colab's system FFmpeg.")
        print(extractor_error)
        frame_dir.mkdir(parents=True, exist_ok=True)
        for partial_frame in frame_dir.glob("frame_*.png"):
            partial_frame.unlink()
        run([
            "ffmpeg", "-hide_banner", "-loglevel", "warning", "-y",
            "-ss", "0", "-t", "219.1", "-i", VIDEO_SOURCE, "-an",
            "-vf", "fps=30", "-pix_fmt", "gray", "-start_number", "0",
            frame_dir / "frame_%05d.png",
        ])

frame_count = len(list(frame_dir.glob("frame_*.png")))
if frame_count != 6573:
    raise RuntimeError(f"Expected 6573 frames, found {frame_count}.")
print(f"Dataset ready: {frame_count} frames in {frame_dir}")


Dataset ready: 6573 frames in /content/bad_apple/prototype_data/full_source_frames


In [45]:
# Fast preflight: catches a stale branch that lacks the zero-anchor implementation.
run(["python", "-m", "unittest", "discover", "-s", "tests", "-p", "test_pipeline.py", "-v"], cwd=PROJECT_DIR)


+ python -m unittest discover -s tests -p test_pipeline.py -v
test_latent_windows_include_the_next_target (test_pipeline.AutoregressiveTests.test_latent_windows_include_the_next_target) ... ok
test_predictor_preserves_latent_grid (test_pipeline.AutoregressiveTests.test_predictor_preserves_latent_grid) ... ok
test_rollout_preserves_context_before_source_cutoff (test_pipeline.AutoregressiveTests.test_rollout_preserves_context_before_source_cutoff) ... ok
test_source_frame_becomes_binary_neuron_targets (test_pipeline.DatasetTests.test_source_frame_becomes_binary_neuron_targets) ... ok
test_border_polarity_detection (test_pipeline.HybridTests.test_border_polarity_detection) ... ok
test_hybrid_rollout_preserves_context (test_pipeline.HybridTests.test_hybrid_rollout_preserves_context) ... ok
test_hybrid_window_contains_rollout_targets (test_pipeline.HybridTests.test_hybrid_window_contains_rollout_targets) ... ok
test_moving_bleed_only_adds_bounded_memory_correction (test_pipeline.HybridTests

In [46]:
# Build the paths and command for the selected budget. This cell only prints; it does not train.
variant = f"anchors_{ANCHORS:03d}"
run_dir = DRIVE_ROOT / "prototype_runs/anchor_budget_ablation" / variant
raw_output_dir = DRIVE_ROOT / "prototype_outputs/anchor_budget_ablation" / f"{variant}_raw"
polarity_run_dir = DRIVE_ROOT / "prototype_runs/anchor_budget_ablation" / f"{variant}_polarity"
final_output_dir = DRIVE_ROOT / "prototype_outputs/anchor_budget_ablation" / f"{variant}_final"

train_command = [
    "python", "prototype.py", "train-hybrid-v42",
    "--anchors", str(ANCHORS),
    "--run-dir", run_dir,
    "--epochs", str(EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--seed", str(SEED),
    "--device", "cuda",
    "--learning-rate", "3e-4",
]
# The memory regularizer is undefined for the zero-anchor control.
if ANCHORS == 0:
    train_command += ["--anchor-loss-weight", "0"]
print("Selected experiment:", variant)
print("Training output:", run_dir)
print("Command:", " ".join(map(str, train_command)))
print("No training has started.")


Selected experiment: anchors_220
Training output: /content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220
Command: python prototype.py train-hybrid-v42 --anchors 220 --run-dir /content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220 --epochs 12 --batch-size 16 --seed 7 --device cuda --learning-rate 3e-4
No training has started.


## Start one training run

Set `START_TRAINING = True` only when the printed budget and paths are correct. The trainer saves `model_last.pt`, `model_best.pt`, `history.json`, and an updated `report.md` after every completed epoch. It does not currently resume an interrupted partial run, so keep the runtime connected through the selected model.

In [47]:
START_TRAINING = True

if not START_TRAINING:
    raise RuntimeError("Training is armed but disabled. Set START_TRAINING = True in this cell.")
if (run_dir / "model_best.pt").exists():
    raise FileExistsError(f"A checkpoint already exists at {run_dir}; refusing to overwrite it.")

run(train_command, cwd=PROJECT_DIR)


+ python prototype.py train-hybrid-v42 --anchors 220 --run-dir /content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220 --epochs 12 --batch-size 16 --seed 7 --device cuda --learning-rate 3e-4
Encoding the temporally canonical source sequence...
Training bleeding-memory v4.2 on cuda: 6573 frames, 6398 windows, 10,946,534 parameters
Scene anchors: [0, 60, 121, 181, 241, 301, 362, 422, 443, 458, 466, 474, 482, 529, 543, 603, 663, 724, 784, 797, 820, 844, 904, 965, 1025, 1060, 1076, 1085, 1102, 1146, 1206, 1220, 1266, 1293, 1326, 1365, 1387, 1447, 1471, 1507, 1516, 1524, 1568, 1578, 1628, 1688, 1714, 1722, 1730, 1749, 1798, 1809, 1854, 1869, 1884, 1892, 1910, 1929, 1990, 2050, 2110, 2171, 2208, 2231, 2246, 2254, 2291, 2307, 2315, 2323, 2351, 2412, 2472, 2495, 2503, 2532, 2593, 2653, 2713, 2724, 2732, 2774, 2820, 2834, 2851, 2862, 2894, 2922, 2930, 2938, 2946, 2954, 3015, 3075, 3135, 3196, 3256, 3272, 3299, 3316, 3376, 3437, 3454, 3462, 3471, 3497, 3505,

## Evaluate and repair polarity

Run this after training completes. It performs the raw rollout, fits the separate polarity spline, and performs the final metrics-only rollout.

In [48]:
checkpoint = run_dir / "model_best.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Training checkpoint not found: {checkpoint}")

run([
    "python", "prototype.py", "rollout-ar",
    "--checkpoint", checkpoint,
    "--data-dir", frame_dir,
    "--output-dir", raw_output_dir,
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--device", "cuda", "--fps", "30", "--no-video",
], cwd=PROJECT_DIR)

run([
    "python", "prototype.py", "fix-polarity",
    "--checkpoint", checkpoint,
    "--target-csv", raw_output_dir / "error_curve.csv",
    "--run-dir", polarity_run_dir,
    "--device", "cuda",
], cwd=PROJECT_DIR)

run([
    "python", "prototype.py", "rollout-ar",
    "--checkpoint", polarity_run_dir / "model_best.pt",
    "--data-dir", frame_dir,
    "--output-dir", final_output_dir,
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--device", "cuda", "--fps", "30", "--no-video",
], cwd=PROJECT_DIR)


+ python prototype.py rollout-ar --checkpoint /content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220/model_best.pt --data-dir /content/bad_apple/prototype_data/full_source_frames --output-dir /content/drive/MyDrive/neural_bad_apple (1)/prototype_outputs/anchor_budget_ablation/anchors_220_raw --batch-size 16 --device cuda --fps 30 --no-video
{
  "checkpoint": "/content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220/model_best.pt",
  "model_type": "hybrid_v4_bleeding_memory",
  "autoencoder_checkpoint": "/content/bad_apple/prototype_runs/basic_full/model_best.pt",
  "frame_count": 6573,
  "fps": 30.0,
  "warmup_frames": 16,
  "source_cutoff_seconds": 0.5,
  "image_size": [
    384,
    512
  ],
  "canonicalize_polarity": true,
  "polarity_tracking_method": "temporal",
  "polarity_switch_penalty": 0.05,
  "mean_teacher_binary_error": 0.13975494414170464,
  "mean_rollout_binary_error": 0.15678724417296414,
  "mean

In [50]:
# Compact result readout for the blog/report table.
import json

summary_path = final_output_dir / "drift_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
result = {
    "anchors": ANCHORS,
    "rollout_error": summary["post_cutoff_mean_rollout_binary_error"],
    "accumulation_gap": summary["post_cutoff_mean_accumulation_gap"],
    "iou": summary["post_cutoff_mean_rollout_iou"],
    "peak_error": summary["peak_rollout_binary_error"],
    "peak_seconds": summary["peak_error_seconds"],
}
print(json.dumps(result, indent=2))
print(f"Full summary: {summary_path}")
print(f"Training report: {run_dir / 'report.md'}")


{
  "anchors": 220,
  "rollout_error": 0.05559335669542292,
  "accumulation_gap": 0.024289758172325378,
  "iou": 0.828956301093002,
  "peak_error": 0.4180857340494792,
  "peak_seconds": 217.16666666666666
}
Full summary: /content/drive/MyDrive/neural_bad_apple (1)/prototype_outputs/anchor_budget_ablation/anchors_220_final/drift_summary.json
Training report: /content/drive/MyDrive/neural_bad_apple (1)/prototype_runs/anchor_budget_ablation/anchors_220/report.md


## Next budget

Recommended order is `0`, `32`, then compare both with the existing local `220` reference. If that establishes a useful trend, run `16`, `55`, and `110`. To start another budget, change `ANCHORS` in the configuration cell and rerun from the path/command cell downward.